In [1]:
import sys
from pathlib import Path

sys.path.append("../")
from src.components.classification.cls_data_ingestion import ClsDataIngestion
from src.components.classification.cls_data_transform import ClsDataTransformation
from src.components.classification.cls_model import ClassifierModel
from src.entity.cls_entity import ClsDataIngestionConfig, ClsTransformationConfig, ClsModelConfig

d:\marchine_learning\Projet\medical_project\medical_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Test the data Ingestion 
clsDataIngestionConfig = ClsDataIngestionConfig(
    ext ='.dicom',
    csv_col = {
        "img": "image_id",
        "label": "lesion_type"
    },
    path_root = Path("D:/marchine_learning/Projet/medical_project/AI-Clinical-Imaging-Assistant/data/vindr-spinexr"),
    multiclass = False,
    paths_img = ("train_images", "test_images"),
    paths_csv =("annotations/train.csv", "annotations/test.csv"),
    samples_rate = (0.05, 0.02),
    seed = 50
)

clsDataIngestion = ClsDataIngestion(clsDataIngestionConfig)
ingestion_result = clsDataIngestion.get_files()

img_files = ingestion_result.images 
labels = ingestion_result.labels
weights = ingestion_result.weights
pos_weigths = ingestion_result.pos_weight

[2026-07-21 00:44:44,420: INFO: cls_data_ingestion: Configured pipeline for Multi-label / Binary classification.]
[2026-07-21 00:44:44,420: INFO: cls_data_ingestion: Successfully ingested 108 files into the train pipeline.]
[2026-07-21 00:44:44,470: INFO: cls_data_ingestion: Configured pipeline for Multi-label / Binary classification.]
[2026-07-21 00:44:44,470: INFO: cls_data_ingestion: Successfully ingested 41 files into the val pipeline.]


In [3]:
# Test the data Transformation
clsTransformationConfig = ClsTransformationConfig(
    image_size = (128, 128),
    cache_rate =(0.01, 0.01),
    cache_num_workers = 2,
    flip = {"prob": 0.2, "axis": 1},
    contrast ={"prob": 0.3, "gamma": (0.7, 1.3)},
    gaussian_noise = {"prob": 0.2, "std": 0.02},
    affine = {"prob": 0.5, "rotate": 15.0, "scale": (0.8, 1.2), "translate": (10, 10), "pad_mode": "border"},
)

clsDataTransformation = ClsDataTransformation(clsTransformationConfig, clsDataIngestionConfig.multiclass)
train_ds, valid_ds = clsDataTransformation.transforms(img_files, labels)

Loading dataset:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-07-21 00:44:45,017: WARNING: dataset: Error while processing tag 00080018: Invalid value for VR UI: '1cb0758b18da36911bad7069a7238558'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.]
[2026-07-21 00:44:45,017: WARNING: dataset: Error while processing tag 0020000D: Invalid value for VR UI: '9a06975c67c46f6986419292f7b3fe7d'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.]
[2026-07-21 00:44:45,017: WARNING: dataset: Error while processing tag 0020000E: Invalid value for VR UI: 'e25e9107cb5dd52735ff68f102e9020c'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.]


Loading dataset: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]

[2026-07-21 00:44:45,143: INFO: cls_data_transform: Dataset successfully initialized. Cache rate: 1.0%]
[2026-07-21 00:44:45,144: INFO: cls_data_transform: Training Dataset pipeline built successfully.]
[2026-07-21 00:44:45,146: INFO: cls_data_transform: Dataset successfully initialized. Cache rate: 1.0%]
[2026-07-21 00:44:45,148: INFO: cls_data_transform: Validation Dataset pipeline built successfully.]


In [4]:
# Test the Clasifier Model
clsModelConfig = ClsModelConfig(
    dropout_rate= (0.1, 0.1), 
    in_chans = 1,
    feature_head = 128,
    model_name = "efficientnet_b0",
    num_classes = 8,
    pretrained = True, 
    uuid_tag ="sssd", 
)

classifierModel = ClassifierModel(clsModelConfig)

classifierModel.eval()
out = classifierModel(train_ds[0]["image"].unsqueeze(0))
out.shape

[2026-07-21 00:44:45,274: INFO: _builder: Loading pretrained weights from Hugging Face hub (timm/efficientnet_b0.ra_in1k)]
[2026-07-21 00:44:45,708: INFO: _client: HTTP Request: HEAD https://huggingface.co/timm/efficientnet_b0.ra_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"]
[2026-07-21 00:44:45,708: INFO: _hub: [timm/efficientnet_b0.ra_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.]
[2026-07-21 00:44:45,731: INFO: _builder: Converted input conv conv_stem pretrained weights from 3 to 1 channel(s)]


torch.Size([1, 8])

In [5]:
## Test train loops
import torch
from torch.utils.data import DataLoader
from src.components.model_trainer import ModelTrainer
from src.entity.train_entity import TrainerConfig, EarlyStoppingConfig
from src.callbacks.cls_callback import EvaluationCbk, LogMetricsCbk
from src.callbacks.callbacks import EarlyStoppingCbk
from accelerate import Accelerator

kwarg_loader = {
    "num_workers": 4,
    "batch_size": 4,
    "pin_memory": True,
    "prefetch_factor": 4,
    "persistent_workers" :True,
    "drop_last" :True,
}

earlyStopingConfig = EarlyStoppingConfig(
    patience = 2,
    min_delta =  1e-7,
    monitor ='loss',
    mode ='min', 
)

trainerConfig = TrainerConfig(
    epochs = 3,
    accumulation_steps = 2,
    lr = 1e-4,
    max_grad_norm = 1.0,
    artifacts_root = Path("D:/marchine_learning/Projet/medical_project/AI-Clinical-Imaging-Assistant/checkpoint/"),
    step_freq_save = 4,
    amp = "",
    compile_model = False,
)

accelerator = Accelerator()
optimizer = torch.optim.AdamW(classifierModel.parameters(), lr=trainerConfig.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=trainerConfig.epochs, eta_min = 1e-7)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weigths, dtype=torch.float32)).to(accelerator.device)

train_loader = DataLoader(train_ds, shuffle=True, **kwarg_loader)
valid_loader = DataLoader(valid_ds, shuffle=False, **kwarg_loader)

In [6]:
from torchmetrics import MetricCollection
from torchmetrics.classification import (MultilabelAccuracy, MultilabelF1Score)

classification_metrics = MetricCollection({
    "accuracy": MultilabelAccuracy(num_labels=8, average="macro"),
    "f1_macro": MultilabelF1Score(num_labels=8, average="macro")
}).to(accelerator.device)


logMetricsCbk = LogMetricsCbk()

earlyStoppingCbk = EarlyStoppingCbk(earlyStopingConfig)
evaluationCbk = EvaluationCbk(criterion , classification_metrics, cls_type = "multilabel",  threshold = 0.50)

modelTrainer = ModelTrainer(accelerator, trainerConfig, model=classifierModel, 
                            optimizer=optimizer, scheduler=scheduler,
                            criterion=criterion, 
                            callbacks=[evaluationCbk, logMetricsCbk, earlyStoppingCbk])

[MLOps Info] Standard eager execution mode active.


In [7]:
modelTrainer.train(train_loader, valid_loader)

Epoch 1/3:   0%|          | 0/27 [00:00<?, ?it/s]d:\marchine_learning\Projet\medical_project\medical_env\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Validation Epoch 1:  90%|█████████ | 9/10 [00:38<00:01,  1.40s/it, batch_loss=1.75] 

[2026-07-21 00:46:23,304: INFO: accelerator: [RANK 0] The used dataset had no length, returning gathered tensors. You should drop the remainder yourself.]


Validation Epoch 1: 100%|██████████| 10/10 [00:38<00:00,  3.81s/it, batch_loss=1.75]


Epoch 1 - Train/Val Loss Ratio: 1.1524 / 1.1065
Epoch 1 - Detailed Validation Metrics: {'accuracy': 0.581250011920929, 'f1_macro': 0.20839379727840424, 'loss': 1.1064746379852295}


Validation Epoch 2:  80%|████████  | 8/10 [00:09<00:02,  1.23s/it, batch_loss=2.07] 

...

Epoch 2 - Train/Val Loss Ratio: 1.2501 / 1.2210
Epoch 2 - Detailed Validation Metrics: {'accuracy': 0.6656249761581421, 'f1_macro': 0.20871488749980927, 'loss': 1.2209546566009521}
--- EarlyStopping Counter updated: 1/2 (Best historical score: 1.106475) ---


Validation Epoch 3:  80%|████████  | 8/10 [00:09<00:02,  1.16s/it, batch_loss=2.08] 

[2026-07-21 00:47:51,541: INFO: accelerator: [RANK 0] The used dataset had no length, returning gathered tensors. You should drop the remainder yourself.]


Validation Epoch 3: 100%|██████████| 10/10 [00:09<00:00,  1.08it/s, batch_loss=2.08]


Epoch 3 - Train/Val Loss Ratio: 1.2773 / 1.1841
Epoch 3 - Detailed Validation Metrics: {'accuracy': 0.734375, 'f1_macro': 0.1697852909564972, 'loss': 1.1840744018554688}
--- EarlyStopping Counter updated: 2/2 (Best historical score: 1.106475) ---
Early stopping criteria met. Terminating training pipeline.
Early stopping condition triggered. Halting at epoch 3.
